In [29]:
import pandas as pd
import json
import warnings
import numpy as np
import os
import numpy as np
import sys



warnings.filterwarnings('ignore')
RANDOM = False


warnings.filterwarnings('ignore')
url =  "../estadistica_stop/ESTADISTICA_DELITO.csv"  #"save3.to_csv("data2.csv",compression='xz', sep='\t', index=False)"
df = pd.read_csv(url)

df = df[df["codcom"].isin(list(codReg2["Codcom"]))]


In [30]:
codReg = localiza[["Codcom","Codreg"]].drop_duplicates()
codReg2 = codReg[codReg["Codreg"] == 13]

In [31]:
# Función para verificar que el número de filas no cambie inesperadamente
def verificar_integridad(df_actual, df_anterior, nombre_paso, debe_crecer=False):
    filas_actuales = len(df_actual)
    filas_anteriores = len(df_anterior)
    
    if debe_crecer:
        if filas_actuales <= filas_anteriores:
            print(f"❌ ERROR EN {nombre_paso}: Se esperaba que las filas aumentaran, pero se mantuvieron o redujeron.")
            print(f"   Antes: {filas_anteriores} | Después: {filas_actuales}")
            sys.exit("Ejecución detenida por inconsistencia de datos.")
    else:
        if filas_actuales != filas_anteriores:
            print(f"❌ ERROR EN {nombre_paso}: El número de filas cambió inesperadamente.")
            print(f"   Antes: {filas_anteriores} | Después: {filas_actuales}")
            print(f"   Esto indica un producto cartesiano (duplicados en el merge).")
            sys.exit("Ejecución detenida por riesgo de MemoryError o datos corruptos.")
    
    print(f"✅ {nombre_paso}: {filas_actuales} líneas (OK)")


# =========================================
# 2. CALCULAR TOTALES POR COMUNA/SEMANA
# =========================================
# Asumimos que 'df' ya está cargado antes de este bloque
lineas_inicio = len(df)

# Totales
totales = (
    df.groupby(['codcom', 'id_semana'], as_index=False)['frecuencia']
      .sum()
)
totales['delito'] = 'Total'

# Dimensión tiempo
dim_tiempo = df[['id_semana', 'semana_detalle', 'fecha']].drop_duplicates()
totales = totales.merge(dim_tiempo, on='id_semana', how='left')

# Asegurar mismas columnas que df
totales = totales.reindex(columns=df.columns, fill_value=np.nan)

# Concatenar
df = pd.concat([df, totales], ignore_index=True)
verificar_integridad(df, pd.DataFrame(index=range(lineas_inicio)), "Concatenar Totales", debe_crecer=True)


# =========================================
# 3. PREPARACIÓN TEMPORAL
# =========================================
df['fecha'] = pd.to_datetime(df['fecha'], errors='coerce')
df['año'] = df['fecha'].dt.year
df['mes'] = df['fecha'].dt.month

# Semana numérica desde semana_detalle (ej: "Semana 05")
df['semana_numero'] = (
    df['semana_detalle']
    .astype(str)
    .str.extract(r'(\d{1,2})')
    .astype(float)
)

# Orden crítico
df = df.sort_values(['delito', 'codcom', 'id_semana']).reset_index(drop=True)


# =========================================
# 4. VARIABLES BASE
# =========================================
df['casos_semana_actual'] = df['frecuencia']
df['casos_semana_anterior'] = (
    df.groupby(['delito', 'codcom'])['frecuencia'].shift(1)
)
df['delta'] = df['casos_semana_actual'] - df['casos_semana_anterior']
# No verificamos integridad aquí porque son transformaciones in-place


# =========================================
# 5. ACUMULADOS
# =========================================
df['acumulado_anual'] = (
    df.groupby(['delito', 'codcom', 'año'])['frecuencia'].cumsum()
)
df['acumulado_total'] = (
    df.groupby(['delito', 'codcom'])['frecuencia'].cumsum()
)

# Acumulado año anterior
df_prev_acum = df[['delito','codcom','año','semana_numero','acumulado_anual']].copy()

# 🔴 CORRECCIÓN CLAVE: Eliminar duplicados antes de cambiar el año para evitar MemoryError
# Si hay múltiples registros para la misma semana/comuna, el merge explotará.
# Como 'cumsum' se calculó por grupo, el valor es el mismo para todos los registros de la clave,
# así que es seguro eliminar duplicados.
df_prev_acum = df_prev_acum.drop_duplicates(subset=['delito','codcom','año','semana_numero'])

df_prev_acum['año'] += 1
df_prev_acum.rename(
    columns={'acumulado_anual':'acumulado_anual_anterior'},
    inplace=True
)

df_len_before_merge = len(df)
df = df.merge(
    df_prev_acum,
    on=['delito','codcom','año','semana_numero'],
    how='left'
)

# Verificar que el merge no multiplicó las filas
verificar_integridad(df, pd.DataFrame(index=range(df_len_before_merge)), "Merge Acumulado Año Anterior")


# =========================================
# 6. MEDIAS MÓVILES
# =========================================
df['media_movil_4s'] = (
    df.groupby(['delito','codcom'])['frecuencia']
      .transform(lambda x: x.rolling(4, min_periods=1).mean())
)

df['media_movil_8s'] = (
    df.groupby(['delito','codcom'])['frecuencia']
      .transform(lambda x: x.rolling(8, min_periods=1).mean())
)


# =========================================
# 7. HISTÓRICOS
# =========================================
df['promedio_hist'] = (
    df.groupby(['delito','codcom'])['frecuencia']
      .transform(lambda x: x.expanding().mean())
)

df['std_hist'] = (
    df.groupby(['delito','codcom'])['frecuencia']
      .transform(lambda x: x.expanding().std())
)

df['max_hist'] = (
    df.groupby(['delito','codcom'])['frecuencia']
      .transform(lambda x: x.expanding().max())
)


# =========================================
# 8. ESTADÍSTICAS AÑO ANTERIOR
# =========================================
df['promedio_hist_anual'] = (
    df.groupby(['delito','codcom','año'])['frecuencia']
      .transform(lambda x: x.expanding().mean())
)

df['std_hist_anual'] = (
    df.groupby(['delito','codcom','año'])['frecuencia']
      .transform(lambda x: x.expanding().std())
)

df['max_hist_anual'] = (
    df.groupby(['delito','codcom','año'])['frecuencia']
      .transform(lambda x: x.expanding().max())
)

stats_prev = df[['delito','codcom','año','semana_numero',
                 'promedio_hist_anual','std_hist_anual','max_hist_anual']].copy()

# 🔴 CORRECCIÓN CLAVE: Eliminar duplicados antes del merge
# Evita el MemoryError: Unable to allocate 19.6 GiB...
stats_prev = stats_prev.drop_duplicates(subset=['delito','codcom','año','semana_numero'])

stats_prev['año'] += 1
stats_prev.rename(columns={
    'promedio_hist_anual':'promedio_hist_anual_prev',
    'std_hist_anual':'std_hist_anual_prev',
    'max_hist_anual':'max_hist_anual_prev'
}, inplace=True)

df_len_before_stats = len(df)
df = df.merge(
    stats_prev,
    on=['delito','codcom','año','semana_numero'],
    how='left'
)

# Verificar integridad
verificar_integridad(df, pd.DataFrame(index=range(df_len_before_stats)), "Merge Stats Año Anterior")

df['promedio_hist_anual'] = df['promedio_hist_anual_prev'].fillna(df['promedio_hist_anual'])
df['std_hist_anual'] = df['std_hist_anual_prev'].fillna(df['std_hist_anual'])
df['max_hist_anual'] = df['max_hist_anual_prev'].fillna(df['max_hist_anual'])

df.drop(columns=[
    'promedio_hist_anual_prev',
    'std_hist_anual_prev',
    'max_hist_anual_prev'
], inplace=True)


# =====================================================
# 9. TENDENCIA Y RACHA
# =====================================================
df['tendencia_corto_plazo'] = np.where(
    df['delta'] > 0, 'Alza',
    np.where(df['delta'] < 0, 'Baja', 'Estable')
)

# Racha de aumentos consecutivos
df['racha'] = (
    (df['delta'] > 0)
    .astype(int)
    .groupby((df['delta'] <= 0).cumsum())
    .cumsum()
)


# =========================================
# 10. MÉTRICAS AVANZADAS
# =========================================
df['var_pct_vs_semana_anterior'] = (
    df['delta'] / df['casos_semana_anterior'].replace(0, np.nan) * 100
)

df['z_score'] = (
    (df['frecuencia'] - df['promedio_hist']) /
    df['std_hist'].replace(0, np.nan)
)

df['z_score_vs_año_anterior'] = (
    (df['frecuencia'] - df['promedio_hist_anual']) /
    df['std_hist_anual'].replace(0, np.nan)
)

df['conclusion_z'] = pd.cut(
    df['z_score'].fillna(0),
    bins=[-np.inf, -2, 2, np.inf],
    labels=['Bajo', 'Normal', 'Alto']
)


# =========================================
# 11. LOCALIZACIÓN Y RANKING
# =========================================
try:
    localiza = pd.read_excel(r"D:\GitHub\LOCALIZA_DB\Localiza Chile (1).xlsx")
    localiza2 = (
        localiza[['Provincia', 'Comuna', 'Región', 'Codcom', 'Codreg']]
        .drop_duplicates()
    )

    df_antes_localiza = len(df)
    df = df.merge(
        localiza2,
        left_on='codcom',
        right_on='Codcom',
        how='left'
    )
    
    # Verificar integridad (Left join no debe aumentar filas)
    verificar_integridad(df, pd.DataFrame(index=range(df_antes_localiza)), "Merge Localización")

    # Ranking comunal regional
    df = df.sort_values(['Codreg', 'delito', 'Codcom', 'id_semana'])

    df['ranking_comunal_regional'] = (
        df.groupby(['Codreg', 'delito', 'id_semana'])['frecuencia']
           .rank(method='dense', ascending=False)
    )

    df['ranking_comunal_regional_semana_anterior'] = (
        df.groupby(['Codreg', 'delito', 'Codcom'])['ranking_comunal_regional']
           .shift(1)
    )

except FileNotFoundError as e:
    print(f"⚠️ Advertencia: No se encontraron archivos de localización. {e}")
    print("   Continuando sin datos de localización...")


# =========================================
# 12. FACTORES POBLACIÓN
# =========================================
try:
    clasePoblacion = pd.read_excel(
        r"C:\Users\limc_\Downloads\Factores Población.xlsx",
        sheet_name="Clase Población"
    )
    factor = pd.read_excel(
        r"C:\Users\limc_\Downloads\Factores Población.xlsx",
        sheet_name="Factores"
    )

    clasePoblacion2 = clasePoblacion[['Codcom', 'Población', 'Clase Población']].copy()
    clasePoblacion2.columns = ['Codcom', 'poblacion_clase', 'clase_poblacion']

    factor2 = factor[['Codcom', 'Año', 'Población', 'Factor Población']].copy()
    factor2.columns = ['Codcom', 'año', 'poblacion', 'factor_poblacion']

    df_antes_poblacion = len(df)
    df = (
        df
        .merge(clasePoblacion2, on='Codcom', how='left')
        .merge(factor2, on=['Codcom', 'año'], how='left')
    )
    
    verificar_integridad(df, pd.DataFrame(index=range(df_antes_poblacion)), "Merge Población")

    # Limpieza final - eliminar columna Codcom si existe
    if 'Codcom' in df.columns:
        df = df.drop(columns=['Codcom'])

except FileNotFoundError as e:
    print(f"⚠️ Advertencia: No se encontraron archivos de población. {e}")
    print("   Continuando sin datos de población...")


# =========================================
# 13. MÁXIMOS HISTÓRICOS Y ALERTAS
# =========================================
# --- CORRECCIÓN ROBUSTA PARA SEMANA MÁXIMO HISTÓRICO ---
idx_max_hist = df.groupby(['delito', 'codcom'])['frecuencia'].idxmax()
# Use .loc to extract exact winning rows
info_maximos = df.loc[idx_max_hist, ['delito', 'codcom', 'id_semana', 'semana_detalle']].copy()
info_maximos.rename(columns={
    'id_semana': 'id_semana_max_hist',
    'semana_detalle': 'semana_detalle_max_hist'
}, inplace=True)

df_antes_maximos = len(df)
df = df.merge(info_maximos, on=['delito', 'codcom'], how='left')
verificar_integridad(df, pd.DataFrame(index=range(df_antes_maximos)), "Merge Máximos Históricos")

# Alertas
df['alerta_aumento_critico'] = (df['z_score'] > 2) & (df['var_pct_vs_semana_anterior'] > 30)
df['alerta_vs_año_anterior'] = (df['z_score_vs_año_anterior'] > 2) & (df['frecuencia'] > df['max_hist_anual'])

# Casos misma semana año anterior
df_prev_casos = df[['delito', 'codcom', 'año', 'semana_numero', 'frecuencia']].copy()
df_prev_casos['año'] = df_prev_casos['año'] + 1
df_prev_casos = df_prev_casos.rename(columns={'frecuencia': 'casos_misma_semana_año_anterior'})
df_prev_casos = df_prev_casos.drop_duplicates(subset=['delito', 'codcom', 'año', 'semana_numero'])

df_antes_prev_casos = len(df)
df = df.merge(df_prev_casos, on=['delito', 'codcom', 'año', 'semana_numero'], how='left')
verificar_integridad(df, pd.DataFrame(index=range(df_antes_prev_casos)), "Merge Casos Misma Semana Año Anterior")

# Casos Mismo Mes Año Anterior
monthly_cases = df.groupby(['delito', 'codcom', 'año', 'mes'])['frecuencia'].sum().reset_index()
monthly_cases.rename(columns={'frecuencia': 'total_casos_mes_real'}, inplace=True)
prev_year_monthly = monthly_cases.copy()
prev_year_monthly['año'] += 1
prev_year_monthly.rename(columns={'total_casos_mes_real': 'casos_mismo_mes_año_anterior'}, inplace=True)

df_antes_monthly = len(df)
df = df.merge(prev_year_monthly, on=['delito', 'codcom', 'año', 'mes'], how='left')
verificar_integridad(df, pd.DataFrame(index=range(df_antes_monthly)), "Merge Casos Mismo Mes Año Anterior")


# =========================================
# 14. TARJETAS COMPLEJAS (T19-T25)
# =========================================

# >>>> T19 y T20: Tipologías Críticas <<<<
# 1. Peor Regional por Semana
if 'ranking_comunal_regional' in df.columns:
    # Crear subset sin totales para este cálculo
    df_delitos = df[df['delito'] != 'Total'].copy()
    
    idx_worst_reg_sem = df_delitos.groupby(['codcom', 'id_semana'])['ranking_comunal_regional'].idxmin()
    worst_reg_sem = df_delitos.loc[idx_worst_reg_sem][['codcom', 'id_semana', 'delito', 'ranking_comunal_regional']].copy()
    worst_reg_sem.rename(columns={'delito': 't19_delito_sem', 'ranking_comunal_regional': 't19_rank_sem'}, inplace=True)
    
    df_antes_t19 = len(df)
    df = df.merge(worst_reg_sem, on=['codcom', 'id_semana'], how='left')
    verificar_integridad(df, pd.DataFrame(index=range(df_antes_t19)), "Merge T19 - Peor Regional")
else:
    print("⚠️ Advertencia: No se puede calcular T19 sin datos de ranking regional")
    df['t19_delito_sem'] = None
    df['t19_rank_sem'] = None

# 2. Peor Nacional por Semana
df['ranking_nacional_semanal'] = df.groupby(['delito', 'id_semana'])['frecuencia'].rank(method='dense', ascending=False)

# Actualizar df_delitos después de agregar la nueva columna
df_delitos = df[df['delito'] != 'Total'].copy()

idx_worst_nac_sem = df_delitos.groupby(['codcom', 'id_semana'])['ranking_nacional_semanal'].idxmin()
worst_nac_sem = df.loc[idx_worst_nac_sem][['codcom', 'id_semana', 'delito', 'ranking_nacional_semanal']].copy()
worst_nac_sem.rename(columns={'delito': 't20_delito_sem', 'ranking_nacional_semanal': 't20_rank_sem'}, inplace=True)

df_antes_t20 = len(df)
df = df.merge(worst_nac_sem, on=['codcom', 'id_semana'], how='left')
verificar_integridad(df, pd.DataFrame(index=range(df_antes_t20)), "Merge T20 - Peor Nacional")

# Shift para semana anterior
df = df.sort_values(['codcom', 'delito', 'id_semana'])
df['t19_delito_ant'] = df.groupby(['delito', 'codcom'])['t19_delito_sem'].shift(1)
df['t19_rank_ant'] = df.groupby(['delito', 'codcom'])['t19_rank_sem'].shift(1)
df['t20_delito_ant'] = df.groupby(['delito', 'codcom'])['t20_delito_sem'].shift(1)
df['t20_rank_ant'] = df.groupby(['delito', 'codcom'])['t20_rank_sem'].shift(1)


# >>>> T21: Concentración Delictual (Pareto) <<<<
top_delitos = df_delitos.sort_values(['codcom', 'id_semana', 'frecuencia'], ascending=[True, True, False])
top_grp = top_delitos.groupby(['codcom', 'id_semana']).head(3)

def agg_top3(x):
    d = {}
    base_sum = df[(df['codcom']==x.name[0]) & (df['id_semana']==x.name[1]) & (df['delito']=='Total')]['frecuencia'].values
    grand_total = base_sum[0] if len(base_sum) > 0 else 1
    i = 1
    for _, row in x.iterrows():
        d[f't21_delito_{i}'] = row['delito']
        d[f't21_val_{i}'] = (row['frecuencia'] / grand_total * 100) if grand_total > 0 else 0
        i += 1
        if i > 3:  # Límite de 3 delitos top
            break
    return pd.Series(d)

top3_info = top_grp.groupby(['codcom', 'id_semana']).apply(agg_top3).reset_index()

df_antes_t21 = len(df)
df = df.merge(top3_info, on=['codcom', 'id_semana'], how='left')
verificar_integridad(df, pd.DataFrame(index=range(df_antes_t21)), "Merge T21 - Top 3 Delitos")


# >>>> T23: Correlación Corto Plazo <<<<
corr_data = []
comunas_unicas = df['codcom'].unique()
print(f"\n🔄 Iniciando cálculo de correlaciones para {len(comunas_unicas)} comunas...")

for i, c in enumerate(comunas_unicas):
    if (i+1) % 35 == 0: 
        print(f"   > Progreso: {i+1}/{len(comunas_unicas)}")
    
    subset = df_delitos[df_delitos['codcom'] == c].copy()
    
    # RANDOM mode logic (si está definida la variable global)
    if 'RANDOM' in globals() and RANDOM:
        corr_data.append({'codcom': c, 't23_d1': 'Random_A', 't23_d2': 'Random_B', 't23_val': round(np.random.rand(), 2)})
        continue

    max_sem = subset['id_semana'].max()
    subset_53 = subset[subset['id_semana'] > (max_sem - 53)]
    
    if len(subset_53) < 20:
        corr_data.append({'codcom': c, 't23_d1': 'Insuf. Datos', 't23_d2': '', 't23_val': 0})
        continue

    pivot = subset_53.pivot_table(index='id_semana', columns='delito', values='frecuencia', fill_value=0)
    if pivot.shape[1] < 2:
        corr_data.append({'codcom': c, 't23_d1': 'Mono-delito', 't23_d2': '', 't23_val': 0})
        continue
        
    corr_matrix = pivot.corr().abs()
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    corr_matrix_masked = corr_matrix.mask(mask)
    s = corr_matrix_masked.unstack()
    so = s.sort_values(ascending=False)
    
    try:
        top_pair = so.index[0]
        val = so.iloc[0]
        corr_data.append({'codcom': c, 't23_d1': top_pair[0], 't23_d2': top_pair[1], 't23_val': round(val, 2)})
    except:
        corr_data.append({'codcom': c, 't23_d1': 'Sin Corr', 't23_d2': '', 't23_val': 0})

df_corr = pd.DataFrame(corr_data)

df_antes_t23 = len(df)
df = df.merge(df_corr, on='codcom', how='left')
verificar_integridad(df, pd.DataFrame(index=range(df_antes_t23)), "Merge T23 - Correlaciones")

print("✅ Correlaciones calculadas exitosamente")


# >>>> T25: Aporte Regional Completo (Anterior vs Actual) <<<<
# Calcular aporte regional si existe la columna Codreg
if 'Codreg' in df.columns:
    # Casos totales por región y semana
    df['casos_semana_regional'] = df.groupby(['Codreg', 'delito', 'id_semana'])['frecuencia'].transform('sum')
    
    # Porcentaje de aporte de la comuna al total regional
    df['aporte_pct_region'] = (df['frecuencia'] / df['casos_semana_regional'] * 100).fillna(0)
    
    # Valores de la semana anterior
    df = df.sort_values(['delito', 'codcom', 'id_semana'])
    df['aporte_pct_region_ant'] = df.groupby(['delito', 'codcom'])['aporte_pct_region'].shift(1)
    df['casos_semana_regional_ant'] = df.groupby(['delito', 'codcom'])['casos_semana_regional'].shift(1)
    
    print("✅ T25 - Aporte Regional calculado")
else:
    print("⚠️ Advertencia: No se puede calcular T25 sin datos de región (Codreg)")
    df['casos_semana_regional'] = None
    df['aporte_pct_region'] = None
    df['aporte_pct_region_ant'] = None
    df['casos_semana_regional_ant'] = None


# =========================================
# 15. RESUMEN FINAL Y VALIDACIÓN
# =========================================
print("\n" + "="*60)
print("📊 RESUMEN FINAL DEL PIPELINE")
print("="*60)
print(f"   ✅ Total de filas procesadas: {len(df):,}")
print(f"   ✅ Total de columnas creadas: {len(df.columns)}")
print(f"   ✅ Memoria utilizada: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"   ✅ Comunas únicas: {df['codcom'].nunique()}")
print(f"   ✅ Delitos únicos: {df['delito'].nunique()}")
print(f"   ✅ Semanas procesadas: {df['id_semana'].nunique()}")
print("="*60)
print("✅ Pipeline ejecutado exitosamente sin errores de memoria ni integridad.")
print("="*60)

# Mostrar primeras columnas para verificación
print("\n📋 Primeras columnas del DataFrame final:")
print(df.columns.tolist()[:20])
if len(df.columns) > 20:
    print(f"   ... y {len(df.columns) - 20} columnas más")


✅ Concatenar Totales: 187492 líneas (OK)
✅ Merge Acumulado Año Anterior: 187492 líneas (OK)
✅ Merge Stats Año Anterior: 187492 líneas (OK)
✅ Merge Localización: 187492 líneas (OK)
✅ Merge Población: 187492 líneas (OK)
✅ Merge Máximos Históricos: 187492 líneas (OK)
✅ Merge Casos Misma Semana Año Anterior: 187492 líneas (OK)
✅ Merge Casos Mismo Mes Año Anterior: 187492 líneas (OK)
✅ Merge T19 - Peor Regional: 187492 líneas (OK)
✅ Merge T20 - Peor Nacional: 187492 líneas (OK)
✅ Merge T21 - Top 3 Delitos: 187492 líneas (OK)

🔄 Iniciando cálculo de correlaciones para 52 comunas...
   > Progreso: 35/52
✅ Merge T23 - Correlaciones: 187492 líneas (OK)
✅ Correlaciones calculadas exitosamente
✅ T25 - Aporte Regional calculado

📊 RESUMEN FINAL DEL PIPELINE
   ✅ Total de filas procesadas: 187,492
   ✅ Total de columnas creadas: 67
   ✅ Memoria utilizada: 316.57 MB
   ✅ Comunas únicas: 52
   ✅ Delitos únicos: 22
   ✅ Semanas procesadas: 164
✅ Pipeline ejecutado exitosamente sin errores de memoria n

In [32]:
df3 = df
# =====================================================
# 11. CÁLCULOS PARA TARJETAS BASE (Cards 1-18)
# =====================================================

# --- A. Proyecciones y Tasas ---
df3['semana_numero_safe'] = df3['semana_numero'].replace(0, 1)
df3['proyeccion_anual'] = (df3['acumulado_anual'] / df3['semana_numero_safe']) * 52
df3['tasa_semanal'] = (df3['frecuencia'] / df3['poblacion']) * 100000
df3['tasa_proyectada_anual'] = (df3['proyeccion_anual'] / df3['poblacion']) * 100000

# --- B. Agregaciones Nacionales ---
grp_nac = df3.groupby(['delito', 'id_semana'])
df3['tasa_proyectada_nacional'] = grp_nac['proyeccion_anual'].transform('sum') / grp_nac['poblacion'].transform('sum') * 100000
df3['tasa_semanal_nacional'] = grp_nac['frecuencia'].transform('sum') / grp_nac['poblacion'].transform('sum') * 100000

# --- C. Agregaciones Regionales ---
grp_reg = df3.groupby(['Codreg', 'delito', 'id_semana'])
df3['tasa_proyectada_regional'] = grp_reg['proyeccion_anual'].transform('sum') / grp_reg['poblacion'].transform('sum') * 100000
df3['tasa_semanal_regional'] = grp_reg['frecuencia'].transform('sum') / grp_reg['poblacion'].transform('sum') * 100000
df3['casos_semana_regional'] = grp_reg['frecuencia'].transform('sum')
df3['aporte_pct_region'] = (df3['frecuencia'] / df3['casos_semana_regional'].replace(0, np.nan)) * 100

# --- D. Rankings ---
df3['ranking_regional_proy_anual'] = df3.groupby(['Codreg', 'delito', 'id_semana'])['proyeccion_anual'].rank(method='dense', ascending=False)
df3['ranking_nacional_semanal'] = df3.groupby(['delito', 'id_semana'])['frecuencia'].rank(method='dense', ascending=False)
df3['ranking_nacional_proy_anual'] = df3.groupby(['delito', 'id_semana'])['proyeccion_anual'].rank(method='dense', ascending=False)

grp_cluster = df3.groupby(['clase_poblacion', 'delito', 'id_semana'])
df3['ranking_cluster_semanal'] = grp_cluster['frecuencia'].rank(method='dense', ascending=False)
df3['ranking_cluster_proy_anual'] = grp_cluster['proyeccion_anual'].rank(method='dense', ascending=False)

# Shifts de Rankings
df3 = df3.sort_values(['Codreg', 'delito', 'codcom', 'id_semana'])
g_temp = df3.groupby(['delito', 'codcom'])
df3['ranking_regional_proy_anual_anterior'] = g_temp['ranking_regional_proy_anual'].shift(1)
df3['ranking_nacional_semanal_anterior'] = g_temp['ranking_nacional_semanal'].shift(1)
df3['ranking_nacional_proy_anual_anterior'] = g_temp['ranking_nacional_proy_anual'].shift(1)
df3['ranking_cluster_semanal_anterior'] = g_temp['ranking_cluster_semanal'].shift(1)

# --- E. Stats Adicionales ---
df3['proyeccion_mes_actual'] = df3['media_movil_4s'] * 4.33
df3['promedio_diario_semanal'] = df3['frecuencia'] / 7
df3['promedio_diario_historico'] = df3['promedio_hist'] / 7

total_semanal_comuna = df3.groupby(['codcom', 'id_semana'])['frecuencia'].transform('sum')
df3['share_delito_semanal'] = (df3['frecuencia'] / total_semanal_comuna.replace(0, np.nan)) * 100

df3.drop(columns=['semana_numero_safe'], inplace=True, errors='ignore')

print("DataFrame Final Listo. Columnas:")
print(df3.columns.tolist())
print(f"\nTotal columnas: {len(df3.columns)}")

DataFrame Final Listo. Columnas:
['delito', 'frecuencia', 'codcom', 'id_semana', 'semana_detalle', 'fecha', 'año', 'mes', 'semana_numero', 'casos_semana_actual', 'casos_semana_anterior', 'delta', 'acumulado_anual', 'acumulado_total', 'acumulado_anual_anterior', 'media_movil_4s', 'media_movil_8s', 'promedio_hist', 'std_hist', 'max_hist', 'promedio_hist_anual', 'std_hist_anual', 'max_hist_anual', 'tendencia_corto_plazo', 'racha', 'var_pct_vs_semana_anterior', 'z_score', 'z_score_vs_año_anterior', 'conclusion_z', 'Provincia', 'Comuna', 'Región', 'Codreg', 'ranking_comunal_regional', 'ranking_comunal_regional_semana_anterior', 'poblacion_clase', 'clase_poblacion', 'poblacion', 'factor_poblacion', 'id_semana_max_hist', 'semana_detalle_max_hist', 'alerta_aumento_critico', 'alerta_vs_año_anterior', 'casos_misma_semana_año_anterior', 'casos_mismo_mes_año_anterior', 't19_delito_sem', 't19_rank_sem', 'ranking_nacional_semanal', 't20_delito_sem', 't20_rank_sem', 't19_delito_ant', 't19_rank_ant', 

In [33]:
# =========================================
# 12. VALIDACIÓN DETALLADA (SIMULACIÓN DASHBOARD)
# =========================================

santiago = df3[(df3['codcom'] == 13101) & (df3['delito'] == 'Total')].sort_values('id_semana')
ultima = santiago.iloc[-1]

print(f"\n=== VALIDACIÓN SANTIAGO {ultima['semana_detalle']} ===\n")
print(f"[T1] Casos Actuales: {ultima['casos_semana_actual']}")
print(f"[T2] Casos Anterior: {ultima['casos_semana_anterior']} (Delta: {ultima['delta']})")
print(f"[T3] Acumulado Anual: {ultima['acumulado_anual']}")
print(f"[T4] Media Movil 4S: {ultima['media_movil_4s']:.1f}")
print(f"[T5] Promedio Histórico: {ultima['promedio_hist']:.1f}")
print(f"[T6] Z-Score: {ultima['z_score']:.2f} ({ultima['conclusion_z']})")
print(f"[T7] Racha: {ultima['racha']} semanas")
print(f"[T8] Max Historico Semana: {ultima['semana_detalle_max_hist']}")
print(f"[T9] Alerta Aumento Critico: {ultima['alerta_aumento_critico']}")
print(f"[T10] Alerta Año Anterior: {ultima['alerta_vs_año_anterior']}")
print(f"[T11] Casos Misma Sem Año Ant: {ultima['casos_misma_semana_año_anterior']}")
print(f"[T12] Casos Mismo Mes Año Ant: {ultima['casos_mismo_mes_año_anterior']}")
print(f"[T13] Ranking Reg Semanal: {ultima['ranking_comunal_regional']}")
print(f"[T14] Ranking Nac Semanal: {ultima['ranking_nacional_semanal']}")
print(f"[T15] Ranking Cluster Semanal: {ultima['ranking_cluster_semanal']}")
print(f"[T16] Proyección Anual: {ultima['proyeccion_anual']:.0f}")
print(f"[T17] Tasa Semanal: {ultima['tasa_semanal']:.1f}")
print(f"[T18] Tasa Proyectada: {ultima['tasa_proyectada_anual']:.1f}")
print(f"[T19] Peor Ranking Regional Actual: {ultima['t19_delito_sem']} (Pos {ultima['t19_rank_sem']:.0f})")
print(f"[T20] Peor Ranking Nacional Actual: {ultima['t20_delito_sem']} (Pos {ultima['t20_rank_sem']:.0f})")
print(f"[T21] Top 1 Delito: {ultima['t21_delito_1']} ({ultima['t21_val_1']:.1f}%)")
print(f"[T23] Correlación Fuerte (53 Sem): {ultima['t23_d1']} vs {ultima['t23_d2']} ({ultima['t23_val']:.2f})")
print(f"[T25] Aporte Regional: {ultima['aporte_pct_region']:.1f}% (Ant: {ultima['aporte_pct_region_ant']:.1f}%)")



=== VALIDACIÓN SANTIAGO SEMANA 05/2026 (del 26/01/2026 al 01/02/2026) ===

[T1] Casos Actuales: 767
[T2] Casos Anterior: 823.0 (Delta: -56.0)
[T3] Acumulado Anual: 3802
[T4] Media Movil 4S: 816.2
[T5] Promedio Histórico: 913.0
[T6] Z-Score: -0.87 (Normal)
[T7] Racha: 0 semanas
[T8] Max Historico Semana: SEMANA 50/2024 (del 09/12/2024 al 15/12/2024)
[T9] Alerta Aumento Critico: False
[T10] Alerta Año Anterior: False
[T11] Casos Misma Sem Año Ant: 887.0
[T12] Casos Mismo Mes Año Ant: 4498.0
[T13] Ranking Reg Semanal: 1.0
[T14] Ranking Nac Semanal: 1.0
[T15] Ranking Cluster Semanal: 1.0
[T16] Proyección Anual: 39541
[T17] Tasa Semanal: 152.5
[T18] Tasa Proyectada: 7860.6
[T19] Peor Ranking Regional Actual: CONSUMO DE ALCOHOL Y DE DROGAS EN LA VÍA PÚBLICA (Pos 1)
[T20] Peor Ranking Nacional Actual: CONSUMO DE ALCOHOL Y DE DROGAS EN LA VÍA PÚBLICA (Pos 1)
[T21] Top 1 Delito: HURTOS (20.9%)
[T23] Correlación Fuerte (53 Sem): ROBOS CON VIOLENCIA E INTIMIDACIÓN vs ROBOS POR SORPRESA (0.79)
[T

In [39]:
for i in df3.iloc[0].keys():
    print(i, df3.iloc[0][i])

delito AMENAZAS CON ARMAS
frecuencia 0
codcom 13101
id_semana 1
semana_detalle SEMANA 01/2023 (del 01/01/2023 al 01/01/2023)
fecha 2023-01-01 00:00:00
año 2023
mes 1
semana_numero 1.0
casos_semana_actual 0
casos_semana_anterior nan
delta nan
acumulado_anual 0
acumulado_total 0
acumulado_anual_anterior nan
media_movil_4s 0.0
media_movil_8s 0.0
promedio_hist 0.0
std_hist nan
max_hist 0.0
promedio_hist_anual 0.0
std_hist_anual nan
max_hist_anual 0.0
tendencia_corto_plazo Estable
racha 0
var_pct_vs_semana_anterior nan
z_score nan
z_score_vs_año_anterior nan
conclusion_z Normal
Provincia Santiago
Comuna Santiago
Región Metropolitana
Codreg 13
ranking_comunal_regional 3.0
ranking_comunal_regional_semana_anterior nan
poblacion_clase 503025
clase_poblacion Más de 100.000
poblacion 482726
factor_poblacion 4.82726
id_semana_max_hist 13
semana_detalle_max_hist SEMANA 13/2023 (del 20/03/2023 al 26/03/2023)
alerta_aumento_critico False
alerta_vs_año_anterior False
casos_misma_semana_año_anterior na

In [35]:
# =========================================
# GUARDADO POR COMUNA (data/stop/{codcom})
# =========================================
import os
output_dir = r'D:\GitHub\STOP_WEB3\web_js\data\stop'
os.makedirs(output_dir, exist_ok=True)

for i in df3["codcom"].unique():
    aux = df3[df3["codcom"] == i]
    aux.to_json(fr'{output_dir}/{i}', orient='records', compression='gzip', indent=None)

print(f"Guardados {df3['codcom'].nunique()} archivos por comuna.")

Guardados 52 archivos por comuna.


In [43]:
df3[df3["clase_poblacion"]  == 'Entre 5.000 y 20.000']["codcom"].unique()

array([13502, 13504, 13505], dtype=int64)

In [41]:
df3["clase_poblacion"].unique()

array(['Más de 100.000', 'Entre 50.000 y 100.000',
       'Entre 20.000 y 50.000', 'Entre 5.000 y 20.000'], dtype=object)

In [66]:
semana = df3[df3["id_semana"] == 164]
semana

,delito,frecuencia,codcom,id_semana,semana_detalle,fecha,año,mes,semana_numero,casos_semana_actual,...,ranking_cluster_semanal,ranking_cluster_proy_anual,ranking_regional_proy_anual_anterior,ranking_nacional_semanal_anterior,ranking_nacional_proy_anual_anterior,ranking_cluster_semanal_anterior,proyeccion_mes_actual,promedio_diario_semanal,promedio_diario_historico,share_delito_semanal
163,AMENAZAS CON ARMAS,1,13101,164,SEMANA 05/2026 (del 26/01/2026 al 01/02/2026),2026-01-26,2026,1,5.0,1,...,3.0,3.0,2.0,2.0,2.0,2.0,7.5775,0.142857,0.216899,0.065189
3771,AMENAZAS CON ARMAS,0,13102,164,SEMANA 05/2026 (del 26/01/2026 al 01/02/2026),2026-01-26,2026,1,5.0,0,...,2.0,3.0,8.0,3.0,8.0,2.0,0.0000,0.000000,0.037456,0.000000
7379,AMENAZAS CON ARMAS,1,13103,164,SEMANA 05/2026 (del 26/01/2026 al 01/02/2026),2026-01-26,2026,1,5.0,1,...,3.0,8.0,7.0,2.0,7.0,2.0,2.1650,0.142857,0.054878,0.471698
10987,AMENAZAS CON ARMAS,1,13104,164,SEMANA 05/2026 (del 26/01/2026 al 01/02/2026),2026-01-26,2026,1,5.0,1,...,3.0,5.0,4.0,3.0,4.0,3.0,4.3300,0.142857,0.058362,0.423729
14595,AMENAZAS CON ARMAS,0,13105,164,SEMANA 05/2026 (del 26/01/2026 al 01/02/2026),2026-01-26,2026,1,5.0,0,...,4.0,9.0,7.0,3.0,7.0,3.0,1.0825,0.000000,0.063589,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173059,VIOLACIONES Y DELITOS SEXUALES,1,13601,164,SEMANA 05/2026 (del 26/01/2026 al 01/02/2026),2026-01-26,2026,1,5.0,1,...,2.0,5.0,13.0,6.0,13.0,3.0,3.2475,0.142857,0.149826,0.806452
176667,VIOLACIONES Y DELITOS SEXUALES,0,13602,164,SEMANA 05/2026 (del 26/01/2026 al 01/02/2026),2026-01-26,2026,1,5.0,0,...,3.0,1.0,11.0,6.0,11.0,2.0,1.0825,0.000000,0.072300,0.000000
180275,VIOLACIONES Y DELITOS SEXUALES,2,13603,164,SEMANA 05/2026 (del 26/01/2026 al 01/02/2026),2026-01-26,2026,1,5.0,2,...,1.0,3.0,15.0,6.0,15.0,2.0,2.1650,0.285714,0.088850,2.941176
183883,VIOLACIONES Y DELITOS SEXUALES,1,13604,164,SEMANA 05/2026 (del 26/01/2026 al 01/02/2026),2026-01-26,2026,1,5.0,1,...,2.0,5.0,13.0,6.0,13.0,3.0,3.2475,0.142857,0.170732,0.617284


In [47]:
semana["t19_delito_sem"].unique()

array(['CONSUMO DE ALCOHOL Y DE DROGAS EN LA VÍA PÚBLICA',
       'OTROS DESÓRDENES PÚBLICOS', 'AMENAZAS CON ARMAS',
       'LESIONES GRAVES', 'INCIVILIDADES', 'LEY DE DROGAS',
       'HOMICIDIOS Y FEMICIDIOS', 'HURTOS', 'AMENAZAS Y RIÑAS',
       'LESIONES MENOS GRAVES'], dtype=object)

In [ ]:
semana["ranking_comunal_regional"]

In [65]:
semana.iloc[0]

delito                                                         AMENAZAS CON ARMAS
frecuencia                                                                      1
codcom                                                                      13101
id_semana                                                                     163
semana_detalle                      SEMANA 04/2026 (del 19/01/2026 al 25/01/2026)
                                                        ...                      
ranking_cluster_semanal_anterior                                              2.0
proyeccion_mes_actual                                                      7.5775
promedio_diario_semanal                                                  0.142857
promedio_diario_historico                                                0.217353
share_delito_semanal                                                     0.060753
Name: 162, Length: 86, dtype: object

In [67]:
acumulador = []

for i in semana["codcom"].unique():
    print(i)
    comuna = semana[semana["codcom"] == i]
    for j in comuna["delito"].unique():
        print(j)
        delito = comuna[comuna["delito"] == j]
        #print(delito.iloc[0]["ranking_comunal_regional"],delito.iloc[0].frecuencia)
        acumulador.append({"codcom":i,"delito":j,"Frecuencia":delito.iloc[0].frecuencia,"ranking_comunal_regional":delito.iloc[0]["ranking_comunal_regional"]})

13101
AMENAZAS CON ARMAS
AMENAZAS Y RIÑAS
CONSUMO DE ALCOHOL Y DE DROGAS EN LA VÍA PÚBLICA
DAÑOS
DELITOS EN CONTEXTO DE VIOLENCIA INTRAFAMILIAR
HOMICIDIOS Y FEMICIDIOS
HURTOS
INCIVILIDADES
LESIONES GRAVES
LESIONES LEVES
LESIONES MENOS GRAVES
LEY DE CONTROL DE ARMAS
LEY DE DROGAS
OTROS DESÓRDENES PÚBLICOS
OTROS ROBOS CON FUERZA EN LAS COSAS
RECEPTACIÓN
ROBOS CON VIOLENCIA E INTIMIDACIÓN
ROBOS DE VEHÍCULOS Y SUS ACCESORIOS
ROBOS EN LUGARES HABITADOS Y NO HABITADOS
ROBOS POR SORPRESA
Total
VIOLACIONES Y DELITOS SEXUALES
13102
AMENAZAS CON ARMAS
AMENAZAS Y RIÑAS
CONSUMO DE ALCOHOL Y DE DROGAS EN LA VÍA PÚBLICA
DAÑOS
DELITOS EN CONTEXTO DE VIOLENCIA INTRAFAMILIAR
HOMICIDIOS Y FEMICIDIOS
HURTOS
INCIVILIDADES
LESIONES GRAVES
LESIONES LEVES
LESIONES MENOS GRAVES
LEY DE CONTROL DE ARMAS
LEY DE DROGAS
OTROS DESÓRDENES PÚBLICOS
OTROS ROBOS CON FUERZA EN LAS COSAS
RECEPTACIÓN
ROBOS CON VIOLENCIA E INTIMIDACIÓN
ROBOS DE VEHÍCULOS Y SUS ACCESORIOS
ROBOS EN LUGARES HABITADOS Y NO HABITADOS
ROBOS POR 

In [59]:
delito

,delito,frecuencia,codcom,id_semana,semana_detalle,fecha,año,mes,semana_numero,casos_semana_actual,...,ranking_cluster_semanal,ranking_cluster_proy_anual,ranking_regional_proy_anual_anterior,ranking_nacional_semanal_anterior,ranking_nacional_proy_anual_anterior,ranking_cluster_semanal_anterior,proyeccion_mes_actual,promedio_diario_semanal,promedio_diario_historico,share_delito_semanal
184046,AMENAZAS CON ARMAS,0,13605,163,SEMANA 04/2026 (del 19/01/2026 al 25/01/2026),2026-01-19,2026,1,4.0,0,...,3.0,7.0,7.0,4.0,7.0,4.0,1.0825,0.0,0.02454,0.0


In [68]:
pd.DataFrame(acumulador).to_excel("ranking_delito.xlsx")